# Email Dump and Image Dataset Exploration

This notebook provides an interactive environment for exploring email dumps and image datasets.

## Setup

In [ ]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from epstein.email_parser import EmailParser
from epstein.image_analyzer import ImageAnalyzer
from epstein.utils import export_to_json, export_to_csv

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Modules loaded successfully")

## Email Analysis

### Load Email Data

In [ ]:
# Initialize email parser
email_parser = EmailParser()

# Parse emails from a directory or MBOX file
# email_parser.parse_directory('/path/to/emails')
# email_parser.parse_mbox('/path/to/mailbox.mbox')

print(f"Loaded {len(email_parser.emails)} emails")

### Email Statistics

In [ ]:
# Get statistics
stats = email_parser.get_email_statistics()

print("Email Statistics:")
print("=" * 50)
for key, value in stats.items():
    print(f"{key}: {value}")

### Convert to DataFrame for Analysis

In [ ]:
# Convert emails to pandas DataFrame
if email_parser.emails:
    df_emails = pd.DataFrame(email_parser.emails)
    
    # Display first few rows
    print(f"DataFrame shape: {df_emails.shape}")
    display(df_emails.head())
else:
    print("No emails loaded yet. Please load emails in the previous cell.")

### Visualize Email Timeline

In [ ]:
if email_parser.emails and 'parsed_date' in df_emails.columns:
    # Filter valid dates
    df_with_dates = df_emails[df_emails['parsed_date'].notna()].copy()
    
    if not df_with_dates.empty:
        # Group by date
        df_with_dates['date_only'] = pd.to_datetime(df_with_dates['parsed_date']).dt.date
        email_counts = df_with_dates.groupby('date_only').size()
        
        # Plot
        plt.figure(figsize=(14, 6))
        email_counts.plot(kind='line', marker='o')
        plt.title('Email Volume Over Time')
        plt.xlabel('Date')
        plt.ylabel('Number of Emails')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print("No valid dates found in emails")
else:
    print("No email data available for visualization")

### Search Emails

In [ ]:
# Search for specific keywords
keyword = "meeting"  # Change this to your keyword
results = email_parser.search_emails(keyword, field='body')

print(f"Found {len(results)} emails containing '{keyword}'")

if results:
    df_results = pd.DataFrame(results)
    display(df_results[['from', 'subject', 'date']].head(10))

## Image Analysis

### Scan Image Directory

In [ ]:
# Initialize image analyzer
image_analyzer = ImageAnalyzer()

# Scan directory for images
# image_analyzer.scan_directory('/path/to/images', recursive=True)

print(f"Found {len(image_analyzer.images)} images")

### Image Statistics

In [ ]:
# Get statistics
img_stats = image_analyzer.get_statistics()

print("Image Statistics:")
print("=" * 50)
for key, value in img_stats.items():
    if key != 'formats':
        print(f"{key}: {value}")

if 'formats' in img_stats:
    print("\nFormats:")
    for fmt, count in img_stats['formats'].items():
        print(f"  {fmt}: {count}")

### Convert to DataFrame

In [ ]:
# Convert images to pandas DataFrame
if image_analyzer.images:
    df_images = pd.DataFrame(image_analyzer.images)
    
    print(f"DataFrame shape: {df_images.shape}")
    display(df_images.head())
else:
    print("No images loaded yet. Please scan a directory in the previous cell.")

### Visualize Image Formats

In [ ]:
if image_analyzer.images and 'format' in df_images.columns:
    format_counts = df_images['format'].value_counts()
    
    plt.figure(figsize=(10, 6))
    format_counts.plot(kind='bar')
    plt.title('Distribution of Image Formats')
    plt.xlabel('Format')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No image data available for visualization")

### Find Duplicates

In [ ]:
# Find duplicate images
duplicates = image_analyzer.find_duplicates()

print(f"Found {len(duplicates)} groups of duplicate images")

if duplicates:
    print("\nShowing first 5 duplicate groups:")
    for i, (hash_val, files) in enumerate(list(duplicates.items())[:5], 1):
        print(f"\nGroup {i} ({len(files)} files):")
        for filepath in files:
            print(f"  - {filepath}")

### Size Distribution

In [ ]:
if image_analyzer.images and 'size_mb' in df_images.columns:
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    df_images['size_mb'].hist(bins=50)
    plt.title('Image Size Distribution')
    plt.xlabel('Size (MB)')
    plt.ylabel('Frequency')
    
    plt.subplot(1, 2, 2)
    df_images['size_mb'].plot(kind='box')
    plt.title('Image Size Box Plot')
    plt.ylabel('Size (MB)')
    
    plt.tight_layout()
    plt.show()
else:
    print("No image size data available")

## Export Results

In [ ]:
# Export email results
if email_parser.emails:
    export_to_json(email_parser.emails, 'email_analysis.json')
    export_to_csv(email_parser.emails, 'email_analysis.csv')
    print("Email data exported")

# Export image results
if image_analyzer.images:
    export_to_json(image_analyzer.images, 'image_analysis.json')
    export_to_csv(image_analyzer.images, 'image_analysis.csv')
    print("Image data exported")